# Befogott vékony rúd nagy lengése — HHT és `modify...` callbackek

**Cél:** síkbeli, nagy elmozdulású/elfordulású lengés számítása kis rugalmas alakváltozás mellett, a LowLevelFEM könyvtár módosítása nélkül.

A példa egy 2D síkfeszültségi kontinuumot használ: `structured_rect_mesh`, másodfokú négyszögelemek, bal oldali befogás, HHT időintegrálás. A rúd egy kezdeti sebességlökést kap az első lineáris hajlítási módus alakjában. A szabad végi célamplitúdó a hossz 20%-a; az elért amplitúdót és alakváltozást a számítás ellenőrzi.

Ez **síkbeli hajlítás**, nem 3D csavarás vagy síkra merőleges lengés. Az anyag ideálisan rugalmas; képlékenységet, kontaktot és gravitációt most nem tartalmaz. Az anyagállandók szemléltető értékek, SI egységekben.

A számításhoz a meglévő LowLevelFEM és Julia standard könyvtárai elegendők. A végső ábrákhoz a mellékelt `plot_results.py` használja a helyben telepített Python/NumPy/Matplotlib csomagokat; az eredmények enélkül is CSV-be és Gmsh-fájlba menthetők.

## 1. Miért geometriailag nemlineáris a feladat?

A **kis alakváltozás** és a **kis elfordulás** külön feltevés. Egy hosszú, vékony rúd nagyot hajolhat, miközben a szálak nyúlása kicsi marad. Nagy merevtest-forgatásnál azonban a lineáris
$\varepsilon=\mathrm{sym}(\nabla_X u)$ fiktív alakváltozást ad.

A teljes Lagrange-féle kinematika:

$$
x=X+u(X,t),\qquad F=I+\nabla_Xu,\qquad
E_G=\frac12(F^TF-I).
$$

Ha $F=R$ tiszta forgatás, akkor $E_G=0$. Ez az objektivitás alapellenőrzése. Minden integrálást a **referenciahálón** végzünk: a Gmsh-csomópontokat nem mozgatjuk időlépésenként.

A Saint-Venant–Kirchhoff-modell a Green–Lagrange-alakváltozás és a második Piola–Kirchhoff-feszültség között lineáris:

$$
S=\mathbb C:E_G,\qquad P=FS,\qquad
\Psi=\tfrac12 E_G:\mathbb C:E_G.
$$

Az anyagi kapcsolat egyszerű marad, a mozgásegyenlet mégis nemlineáris a kinematika miatt. A modell kis **rugalmas** alakváltozásokra használható; nem általános nagy nyúlású anyagmodell.

A síkfeszültségi redukcióban $S_{33}=0$:

$$
\begin{bmatrix}S_{11}\\S_{22}\\S_{12}\end{bmatrix}
=\frac{E_Y}{1-\nu^2}
\begin{bmatrix}1&\nu&0\\\nu&1&0\\0&0&(1-\nu)/2\end{bmatrix}
\begin{bmatrix}E_{11}\\E_{22}\\2E_{12}\end{bmatrix}.
$$

Itt $E_Y$ a Young-modulus. A vastagsági alakváltozás $E_{33}=-\nu(E_{11}+E_{22})/(1-\nu)$; a térfogati arány diagnosztikája ezt is figyelembe veszi.

Elméleti háttér: [COMSOL — Total Lagrangian formulation](https://doc.comsol.com/6.4/doc/com.comsol.help.sme/sme_ug_theory.06.008.html), [lineáris rugalmasság geometriai nemlinearitással](https://www.comsol.com/blogs/modeling-linear-elastic-materials-how-difficult-can-it-be/), [FEniCS — energia, reziduum és konzisztens tangens](https://docs.fenicsproject.org/dolfinx/main/cpp/demos/demo_hyperelasticity.html). Az utóbbi más anyagmodellt használ, az energia–belső erő–tangens felépítés ugyanaz.

## 2. Mit kapnak a HHT callbackek?

A fizikai egyenlet:

$$
M_0\ddot u+C\dot u+f_{\mathrm{int}}(u)=f_{\mathrm{ext}}(t).
$$

A belső erő és a konzisztens tangens ugyanabból az energiából származik:

$$
f_{\mathrm{int}}=\frac{\partial\Pi_{\mathrm{int}}}{\partial u},\qquad
K_T=\frac{\partial f_{\mathrm{int}}}{\partial u}
=K_{\mathrm{mat}}+K_{\mathrm{geo}}.
$$

A HHT `A0*u-rhs` alakját így tesszük azonossá a fizikai reziduummal:

| Callback | Visszaadott érték |
| --- | --- |
| `modifyA0` | $K_T(u)$ |
| `modifyRHS` | $f_{\mathrm{ext}}-f_{\mathrm{int}}(u)+K_T(u)u$ |
| `modifyA2` | `nothing`: a referenciakonfiguráció konzisztens tömegmátrixa állandó. |
| `modifyA1` | `nothing`: itt `C=0`, nincs külön fizikai csillapítás. |

**Önmagában egy `K(u)` frissítése nem elegendő:** általában $f_{\mathrm{int}}(u)\ne K_T(u)u$. A geometriai merevség önmagában sem ad teljes nagyelfordulású modellt.

A callbackek **próbaállapotokon**, egy időlépésen belül többször futnak. Az alábbi cache csak egy közös összeszerelés eredményét adja át A0-ból RHS-be; nem anyagtörténetet halmoz. Az összes mátrix és mező globális, teljes térben marad; a befogást a HHT alkalmazza.

In [12]:
repo = "/home/perebal/Dokumentumok/GitHub/LowLevelFEM.jl"
@assert isfile(joinpath(repo,"Project.toml")) "Állítsd be a LowLevelFEM projekt útvonalát."
repo in LOAD_PATH || pushfirst!(LOAD_PATH,repo)  # nincs Pkg-művelet vagy forrásmódosítás
using LowLevelFEM, LinearAlgebra, SparseArrays, Printf

output_dir = joinpath(pwd(),"results")
mkpath(output_dir)
RUN_GUI = false
RUN_REFINEMENT = true

true

## 3. Geometria és diszkretizáció

A bal szélen minden elmozduláskomponens nulla; külön forgásmező nem kell egy 2D kontinuumhoz. A másodfokú elem az alappéldában egy réteget ad a magasság mentén. A `cells_through_height=2` már térbeli finomítás; a valóban konvergált megoldáshoz ezt is ellenőrizni kell.

A `structured_rect_mesh` a `dx`, `dy` közül a kisebbet használja közös célméretként, ezért a magassági finomítás a hosszirányt is finomítja. A hálógenerátor törli az aktuális Gmsh-modellt; ezt külön notebook-kernelben célszerű futtatni.

In [13]:
L, h, b = 1.0, 0.04, 0.01       # hossz, magasság, referenciavastagság [m]
Eyoung, ν, ρ = 1.0e6, 0.3, 1000.0  # [Pa], [-], [kg/m³]
cells_through_height = 1
A_target = 0.20L                  # sebességlökésből becsült, nem előírt amplitúdó
α_hht = -0.05                     # algoritmikus, nem anyagi csillapítás
samples_per_period = 200
number_of_periods = 2

structured_rect_mesh(lx=L,ly=h,y0=-h/2,
    dx=h/cells_through_height,dy=h/cells_through_height,order=2)
mat = Material("body";E=Eyoung,ν=ν,ρ=ρ)
U = Field([mat];type=:VectorField,dim=2,fieldName=:u)
support = [BoundaryCondition("left";field=U,ux=0.0,uy=0.0)]

node_tags, coordinates, _ = gmsh.model.mesh.getNodes(-1,-1,false,false)
X = zeros(2,U.non)
for (i,node) in enumerate(node_tags)
    X[:,Int(node)] .= coordinates[3i-2:3i-1]
end
right_y = sort(constrainedDoFs(U,[BoundaryCondition("right";field=U,uy=0.0)]))
right_x = right_y .- 1
fixed = constrainedDoFs(U,support)

nodal_field(a) = VectorField(Matrix{Float64}[],reshape(copy(a),:,1),
                            [0.0],Int[],1,:v2D,U)
@printf("Csomópont: %d; elmozdulási DOF: %d; L/h: %.1f\n",U.non,ndofs(U),L/h)

Csomópont: 153; elmozdulási DOF: 306; L/h: 25.0


## 4. A 2D nemlineáris operátor összeszerelése

A következő felhasználói függvény közvetlenül a Gauss-pontokban képezi $F,E_G,S,P$-t, a belső erőt és a két tangensrészt. Nem vetítjük előbb csomópontokra a feszültséget vagy a deformációs gradienst; így az erő és a tangens ugyanannak a diszkrét energiának a deriváltja.

A referencia-deriváltak és a ritka összeszerelés indexei egyszer készülnek el. A Gauss-rend a StVK nemlineáris energiájához igazodik; Q2 elemeknél itt `Gauss9` használatos. A kód az egyetlen `body` anyagcsoporttal megadott példára készült.

A csomag jelenlegi `materialTangentMatrix` és `internalForceVector` általános rutinjai 3D-re korlátozottak, ezért ezt a 2D operátort a notebook definiálja. Nem hívunk nem létező 2D overloadot.

In [14]:
# Felhasználói szintű 2D Total Lagrange / Saint-Venant–Kirchhoff operátor.
# Minden integrálás a referenciahálón történik; a Gmsh-hálót nem mozgatjuk.

function prepare_tl2d(U; thickness, young, poisson)
    @assert U.dim == U.pdim == 2
    @assert thickness > 0 && young > 0 && -1 < poisson < 0.5
    gmsh.model.setCurrent(U.name)
    Dps = young/(1-poisson^2) .* [1.0 poisson 0.0; poisson 1.0 0.0; 0.0 0.0 (1-poisson)/2]
    elements = []
    rows, cols = Int[], Int[]
    for (dim, entity) in gmsh.model.getEntitiesForPhysicalName("body")
        @assert dim == 2
        types, tags, connectivity = gmsh.model.mesh.getElements(2, entity)
        for k in eachindex(types)
            et = types[k]
            _, _, order, nn, _, _ = gmsh.model.mesh.getElementProperties(et)
            # A StVK energia negyedfokú grad(u)-ban: Q2-nél Gauss9 (5×5).
            ip, weights = gmsh.model.mesh.getIntegrationPoints(et, "Gauss$(4order+1)")
            _, rawgrad, _ = gmsh.model.mesh.getBasisFunctions(et, ip, "GradLagrange")
            dN = reshape(rawgrad, 3, nn, length(weights))
            for (e, tag) in enumerate(tags[k])
                nodes = Int.(connectivity[k][(e-1)*nn+1:e*nn])
                @assert all(n -> 1 <= n <= U.non, nodes)
                dofs = vec(permutedims(hcat(2 .* nodes .- 1, 2 .* nodes)))
                jac, determinants, _ = gmsh.model.mesh.getJacobian(tag, ip)
                G = zeros(2, nn, length(weights))
                W = thickness .* weights .* determinants
                @assert all(>(0), W)
                for q in eachindex(weights)
                    J = reshape(jac[9q-8:9q], 3, 3)
                    G[:, :, q] .= (J' \ dN[:, :, q])[1:2, :]
                end
                firstentry = length(rows)+1
                append!(rows, repeat(dofs, length(dofs)))
                append!(cols, repeat(dofs; inner=length(dofs)))
                entries = firstentry:length(rows)
                push!(elements, (; nodes, dofs, G, W, entries))
            end
        end
    end
    return (; U, Dps, poisson, elements, rows, cols, ndof=ndofs(U))
end

function tl2d_response(model, u; tangent=true)
    @assert length(u) == model.ndof
    fint = zeros(model.ndof)
    values = tangent ? zeros(length(model.rows)) : Float64[]
    energy, maxstrain, minJ = 0.0, 0.0, Inf
    for el in model.elements
        nn, ne = length(el.nodes), length(el.dofs)
        ue = reshape(u[el.dofs], 2, nn)
        fe = zeros(ne)
        Ke = tangent ? zeros(ne, ne) : zeros(0,0)
        B = tangent ? zeros(3, ne) : zeros(0,0)
        DB = tangent ? zeros(3, ne) : zeros(0,0)
        for q in eachindex(el.W)
            G = @view el.G[:, :, q]
            F = Matrix{Float64}(I, 2, 2) + ue*G'
            J = det(F)
            J > 0 || error("Invertált síkbeli elem: det(F)=$J. Csökkentsd a Δt/amplitúdó értékét.")
            Egl = (F'*F - I)/2
            e = [Egl[1,1], Egl[2,2], 2*Egl[1,2]]  # engineering shear = 2E12
            s = model.Dps*e
            S = [s[1] s[3]; s[3] s[2]]
            Ppk = F*S
            w = el.W[q]
            fe .+= w .* vec(Ppk*G)
            energy += w*dot(e,s)/2
            # A 3D StVK síkfeszültségi eliminációja: S33=0.
            E33 = -model.poisson/(1-model.poisson)*tr(Egl)
            1+2*E33 > 0 || error("Nem megengedett vastagsági nyúlás a síkfeszültségi modellben.")
            midpoint = tr(Egl)/2
            radius = hypot((Egl[1,1]-Egl[2,2])/2,Egl[1,2])
            maxstrain = max(maxstrain,abs(midpoint+radius),abs(midpoint-radius),abs(E33))
            minJ = min(minJ,J*sqrt(1+2*E33))
            if tangent
                for a in 1:nn
                    gx, gy = G[1,a], G[2,a]
                    B[1,2a-1], B[1,2a] = F[1,1]*gx, F[2,1]*gx
                    B[2,2a-1], B[2,2a] = F[1,2]*gy, F[2,2]*gy
                    B[3,2a-1] = F[1,1]*gy + F[1,2]*gx
                    B[3,2a]   = F[2,1]*gy + F[2,2]*gx
                end
                mul!(DB,model.Dps,B)
                mul!(Ke,B',DB,w,1.0)            # anyagi tangens: B' D B
                geo = G'*S*G
                for a in 1:nn, b in 1:nn
                    Ke[2a-1,2b-1] += w*geo[a,b] # geometriai tangens
                    Ke[2a,2b]     += w*geo[a,b]
                end
            end
        end
        fint[el.dofs] .+= fe
        tangent && (values[el.entries] .= vec(Ke))
    end
    K = tangent ? sparse(model.rows,model.cols,values,model.ndof,model.ndof) : nothing
    return (; fint,K,energy,maxstrain,minJ)
end

function tl2d_callbacks(model)
    # Számítási cache; nem elfogadott anyagtörténet. A0 mindig RHS előtt fut.
    pending = Ref{Any}(nothing)
    evaluations = Ref(0)
    maxiteration = Ref(0)
    modifyA0 = (A,state) -> begin
        U = only(state.problems)
        u = vec(only(state.fields).a)
        response = tl2d_response(model,u)
        pending[] = (; u=copy(u),fint=response.fint)
        evaluations[] += 1
        maxiteration[] = max(maxiteration[],state.iteration)
        SystemMatrix(response.K,U)
    end
    modifyRHS = (rhs,state) -> begin
        u = vec(only(state.fields).a)
        cached = pending[]
        @assert cached !== nothing && cached.u == u
        # Kt*u - (fint - fext): ugyanaz a fizikai reziduum, konzisztens Newton-tangens.
        rhs.a[:,1] .+= state.operators.A0.A*u - cached.fint
        rhs
    end
    return (; modifyA0,modifyRHS,evaluations,maxiteration)
end

tl2d_callbacks (generic function with 1 method)

In [15]:
model = prepare_tl2d(U;thickness=b,young=mat.E,poisson=mat.ν)
zero_state = zeros(ndofs(U))
zero_response = tl2d_response(model,zero_state)
K0 = SystemMatrix(zero_response.K,U)
M = (ρ*b)*∫(U⋅U)                # állandó, konzisztens referenciatömeg
C = 0.0*M                       # fizikai csillapítás ebben a példában nincs
K_dsl = b*∫(SymGrad(U)⋅D(:PlaneStress,mat)⋅SymGrad(U))
@printf("Négyszögelemek: %d\n",length(model.elements))

Négyszögelemek: 25


## 5. A modell alapellenőrzései

1. A zérusállapotbeli tangens egyezzen meg a lineáris DSL-merevséggel.
2. Egy 0,8 radiános merevtest-forgatás adjon nulla alakváltozást és belső erőt.
3. A belső erő véges differenciás irányderiváltja egyezzen $K_Td$-vel.
4. Az energia deriváltja egyezzen a belső erő munkájával; a tangens legyen szimmetrikus.
5. A referencia-tömegmátrix adja vissza a geometria teljes tömegét.

Ezek szükséges modelltesztek; az idő- és hálófinomítás külön vizsgálat.

In [16]:
linear_error = norm(K0.A-K_dsl.A)/norm(K_dsl.A)
@assert linear_error < 1e-10

rotation = [cos(0.8) -sin(0.8);sin(0.8) cos(0.8)]
rigid_u = vec((rotation-I)*X .+ [0.3,-0.2])
rigid = tl2d_response(model,rigid_u)
@assert rigid.maxstrain < 1e-12
@assert norm(rigid.fint)/(Eyoung*b*h) < 1e-10
@assert abs(rigid.minJ-1) < 1e-12

u_test = rigid_u + 0.002L .* sin.(collect(1:ndofs(U)))
direction = cos.(collect(1:ndofs(U)))
direction /= norm(direction)
ε = 1e-7L
test_response = tl2d_response(model,u_test)
plus  = tl2d_response(model,u_test+ε*direction;tangent=false)
minus = tl2d_response(model,u_test-ε*direction;tangent=false)
tangent_error = norm((plus.fint-minus.fint)/(2ε)-test_response.K*direction)/
                norm(test_response.K*direction)
energy_error = abs((plus.energy-minus.energy)/(2ε)-dot(test_response.fint,direction))/
               max(abs(dot(test_response.fint,direction)),eps(Float64))
symmetry_error = norm(test_response.K-test_response.K')/norm(test_response.K)
@assert tangent_error < 1e-6
@assert energy_error < 1e-6
@assert symmetry_error < 1e-12

translation = zeros(ndofs(U));translation[1:2:end] .= 1.0
@assert isapprox(dot(translation,M.A*translation),ρ*L*h*b;rtol=1e-10)
@printf("Lineáris egyezés: %.3e; forgatás max.|E|: %.3e\n",linear_error,rigid.maxstrain)
@printf("Tangenshiba: %.3e; energiaderivált-hiba: %.3e; szimmetriahiba: %.3e\n",
        tangent_error,energy_error,symmetry_error)

Lineáris egyezés: 3.151e-15; forgatás max.|E|: 1.788e-14
Tangenshiba: 2.295e-09; energiaderivált-hiba: 3.418e-09; szimmetriahiba: 6.912e-17


## 6. Gerjesztés és időlépés

A rúd deformálatlan állapotból indul, és első hajlítási módusú kezdeti sebességet kap:

$$
u_0=0,\qquad v_0=A_{\mathrm{target}}\omega_1\varphi_1,
$$

ahol a módusalak szabad végi átlagos függőleges komponense 1. Az $A_{\mathrm{target}}$ a lineáris közelítés amplitúdója; a nemlineáris csúcsérték ettől eltérhet.

A HHT lineáris feltétel nélküli stabilitása nem pontossági garancia a nemlineáris feladatra. Az alappélda `T₁/200` lépést használ; a következő szakasz `Δt/2`-vel is lefuttatja. A gyors axiális/keresztirányú módusokhoz és lokális feszültségcsúcsokhoz külön felbontásvizsgálat kell. Nagy elfordulásnál a lineáris sajátperiódus csak kiinduló időskála.

In [17]:
mode_field = solveEigen(Symmetric(K0),Symmetric(M);
    n=1,solver=:direct,support=support,normalize=:mass)
ω1 = sqrt(eigenValue(mode_field))
T1 = 2π/ω1
mode = copy(mode_field.a[:,1])
mode ./= sum(mode[right_y])/length(right_y)
u0 = nodal_field(zero_state)
v0 = nodal_field(A_target*ω1*mode)
f0 = nodal_field(zero_state)
Δt = T1/samples_per_period
n = number_of_periods*samples_per_period+1

ω_EB = 1.875104068711961^2*sqrt(Eyoung*(b*h^3/12)/(ρ*b*h*L^4))
@assert abs(ω1/ω_EB-1) < 0.05
@printf("ω₁ = %.8g rad/s; T₁ = %.8g s; Euler–Bernoulli eltérés = %.3f%%\n",
        ω1,T1,100abs(ω1/ω_EB-1))
@printf("Δt = %.8g s; minták = %d; kezdeti szabad végi sebesség = %.6g m/s\n",
        Δt,n,A_target*ω1)

ω₁ = 1.2837836 rad/s; T₁ = 4.8942713 s; Euler–Bernoulli eltérés = 0.007%
Δt = 0.024471357 s; minták = 401; kezdeti szabad végi sebesség = 0.256757 m/s


## 7. HHT futtatás a két callbackkel

A callbackekből visszaadott mátrix nincs peremfeltételekkel módosítva. A HHT végzi a befogás kezelését és az időlépésen belüli Newton-iterációt. A példában a tangens teljes `SystemMatrix`, ezért a robusztus direkt LU-megoldást kérjük.

Az első futtatás fordítási időt is tartalmazhat. A `maxiteration` csak diagnosztika: a callback kiértékelése nem azonos egy új elfogadott időlépéssel.

In [ ]:
hooks = tl2d_callbacks(model)
seconds = @elapsed u,v = HHT(K0,M,C,f0,u0,v0,n,Δt;
    support=support,solver=:lu,α=α_hht,
    modifyA0=hooks.modifyA0,modifyRHS=hooks.modifyRHS,
    nonlinearoptions=(maxiter=20,reltol=1e-9,abstol=1e-11))

linear_seconds = @elapsed u_linear,v_linear = HHT(K0,M,C,f0,u0,v0,n,Δt;
    support=support,solver=:lu,α=α_hht)
@printf("Nemlineáris futás: %.2f s; callback-összeszerelések: %d; max. korrekció: %d\n",
        seconds,hooks.evaluations[],hooks.maxiteration[])
@printf("Lineáris referencia: %.2f s\n",linear_seconds)

## 8. Időlépés-felezés

Az összevetésben azonos a háló, kezdeti állapot, HHT-paraméter és a végidő. A finom futás minden második időpontja a durva futáséval egyezik. Ez az alappélda szabad végi elmozdulásának időbeli felbontását ellenőrzi; nem igazolja automatikusan a teljes feszültségmező vagy a háló konvergenciáját.

In [ ]:
if RUN_REFINEMENT
    fine_hooks = tl2d_callbacks(model)
    fine_seconds = @elapsed u_fine,v_fine = HHT(K0,M,C,f0,u0,v0,2*(n-1)+1,Δt/2;
        support=support,solver=:lu,α=α_hht,
        modifyA0=fine_hooks.modifyA0,modifyRHS=fine_hooks.modifyRHS,
        nonlinearoptions=(maxiter=20,reltol=1e-9,abstol=1e-11))
    tip_y = vec(sum(u.a[right_y,:];dims=1))/length(right_y)
    fine_tip_y = vec(sum(u_fine.a[right_y,:];dims=1))/length(right_y)
    step_error = maximum(abs.(tip_y-fine_tip_y[1:2:end]))/maximum(abs,tip_y)
    @assert step_error < 0.01
    @printf("Δt/2: %.2f s; relatív szabad végi eltérés: %.5f%%\n",fine_seconds,100step_error)
else
    u_fine,v_fine = u,v
    step_error = NaN
end

Δt/2: 60.90 s; relatív szabad végi eltérés: 0.09519%


## 9. Fizikai diagnosztika és eredménymentés

Külön mérjük a nagy elmozdulást és a kicsi alakváltozást. A `maxstrain` a három fő Green–Lagrange-alakváltozás legnagyobb abszolút értéke (a visszaszámolt vastagsági komponenssel együtt). A `minJ` a teljes 3D térfogati arány minimuma.

`C=0` mellett is lehet numerikus energiaváltozás. A nemlineáris Newmark/HHT nem garantál pontos energiamegmaradást; `α<0` algoritmikus csillapítást vezet be. A görbe diagnosztika, nem az anyagi energiaelnyelés modellje.

In [ ]:
tip_y = vec(sum(u.a[right_y,:];dims=1))/length(right_y)
tip_x = vec(sum(u.a[right_x,:];dims=1))/length(right_x)
linear_tip_y = vec(sum(u_linear.a[right_y,:];dims=1))/length(right_y)
linear_tip_x = vec(sum(u_linear.a[right_x,:];dims=1))/length(right_x)
kinetic,strain_energy,max_green,min_j = (zeros(n) for _ in 1:4)
for j in 1:n
    response = tl2d_response(model,@view(u.a[:,j]);tangent=false)
    kinetic[j] = dot(v.a[:,j],M.A*v.a[:,j])/2
    strain_energy[j] = response.energy
    max_green[j] = response.maxstrain
    min_j[j] = response.minJ
end
total_energy = kinetic+strain_energy
@assert all(isfinite,total_energy)
@assert maximum(abs,u.a[fixed,:]) < 1e-12
@assert maximum(abs,v.a[fixed,:]) < 1e-12
@assert maximum(abs,tip_y) > 0.1L
@assert maximum(max_green) < 0.03
@assert minimum(min_j) > 0.9
@assert maximum(total_energy)/first(total_energy) < 1.01
@printf("Max. |uᵧ,tip| = %.6g m = %.3f%% L = %.3f h\n",
        maximum(abs,tip_y),100maximum(abs,tip_y)/L,maximum(abs,tip_y)/h)
@printf("Max. fő |E| = %.4f%%; min J = %.8f; E_vég/E_kezdet = %.8f\n",
        100maximum(max_green),minimum(min_j),last(total_energy)/first(total_energy))

function write_csv(path,header,data)
    open(path,"w") do io
        println(io,join(header,","))
        for row in eachrow(data);println(io,join(row,","));end
    end
end
write_csv(joinpath(output_dir,"history.csv"),
    ["time","tau","tip_y_over_L","tip_x_over_L","linear_tip_y_over_L","linear_tip_x_over_L",
     "kinetic_energy","strain_energy","total_energy","max_green_strain","min_J"],
    hcat(u.t,u.t/T1,tip_y/L,tip_x/L,linear_tip_y/L,linear_tip_x/L,
         kinetic,strain_energy,total_energy,max_green,min_j))
fine_tip_y = vec(sum(u_fine.a[right_y,:];dims=1))/length(right_y)
write_csv(joinpath(output_dir,"refined.csv"),["time","tau","tip_y_over_L"],
          hcat(u_fine.t,u_fine.t/T1,fine_tip_y/L))

center = sort(findall(i->abs(X[2,i])<1e-10h,1:U.non);by=i->X[1,i])
@assert !isempty(center)
shape_rows = Vector{Vector{Float64}}()
for (snapshot,τ) in enumerate([0.0,0.25,0.5,0.75,1.0])
    j = argmin(abs.(u.t .- τ*T1))
    for node in center
        push!(shape_rows,[snapshot,u.t[j]/T1,
              (X[1,node]+u.a[2node-1,j])/L,(X[2,node]+u.a[2node,j])/L])
    end
end
write_csv(joinpath(output_dir,"shapes.csv"),["snapshot","tau","x_over_L","y_over_L"],
          reduce(vcat,permutedims.(shape_rows)))
println("CSV-eredmények: ",output_dir)

Max. |uᵧ,tip| = 0.198146 m = 19.815% L = 4.954 h
Max. fő |E| = 1.2406%; min J = 0.99327077; E_vég/E_kezdet = 0.99989685
CSV-eredmények: /home/perebal/Dokumentumok/GitHub/LowLevelFEM.jl/examples/dev/transient/results


## 10. Ábrák és Gmsh-animáció

Az ábrázoló kód csak a mentett CSV-ket olvassa. A deformált geometriát **1:1 léptékkel** mutatjuk: itt a lineáris példákban használt ezres elmozdulásnagyítás félrevezető lenne.

In [ ]:
plotter = joinpath(pwd(),"plot_results.py")
if isfile(plotter) && Sys.which("python3") !== nothing
    run(`python3 $plotter $output_dir`)
    display(MIME"image/png"(),read(joinpath(output_dir,"cantilever_large_rotation.png")))
else
    println("A számítás elkészült. Az ábrákhoz futtasd a mellékelt plot_results.py programot.")
end

A számítás elkészült. Az ábrákhoz futtasd a mellékelt plot_results.py programot.


In [ ]:
view_tag = showDoFResults(u;name="Nagyelfordulású HHT — valós lépték",factor=1.0,visible=true)
gmsh.view.write(view_tag,joinpath(output_dir,"cantilever_HHT.msh"))
if RUN_GUI
    openPostProcessor()
end
println("Gmsh eredmény: ",joinpath(output_dir,"cantilever_HHT.msh"))

Gmsh eredmény: /home/perebal/Dokumentumok/GitHub/LowLevelFEM.jl/examples/dev/transient/results/cantilever_HHT.msh


## 11. Hogyan módosítsd a kísérletet?

- **Amplitúdó:** változtasd `A_target`-et, de nézd a főalakváltozásokat és `J`-t is. A kis rugalmas tartományt az anyag és az igényelt pontosság alapján kell megítélni; a kis alakváltozás nem jelent automatikusan folyáshatár alatti feszültséget.
- **Hálófinomítás:** állítsd `cells_through_height=2`-re, majd indítsd újra a geometriától. Az összes Field/Problem/cache ehhez az új hálóhoz újra kell készüljön. A 2T₁-es időablak helyett több periódus és több háló kell egy frekvencia/amplitúdó tanulmányhoz.
- **Elhúzás–elengedés:** előbb oldd meg a statikus `fint(u)=fpluck` egyenletet ugyanazzal az erővel/tangenssel, terheléslépcsőkkel és Newton-iterációval; ennek megoldása legyen U0, V0 pedig nulla. Nagy kitérésre ne pusztán a lineáris statikus elmozdulást skálázd fel, mert az hamis nyúlási energiát vihet be.
- **Időfüggő külső gerjesztés:** f0 helyett készíts egy n oszlopos terhelésmezőt, például egy `∫(U⋅[0.0,1.0];Γ="right")` terhelésalakból, a vastagság és az amplitúdó helyes szorzójával. A `modifyRHS` a kapott eredeti mintához hozzáadja a Newton-korrekciót, így a callback változatlan maradhat. Finomításkor a terhelést is újra kell mintavételezni.
- **Fizikai csillapítás:** először az anyagi csillapítási modellt válaszd meg. Egy állandó `η*M` csillapítás egyszerű globális sebességellenállás, nem objektív belső viszkoelasztikus anyagtörvény. Ne azonosítsd a HHT α-t az anyagi csillapítással.
- **Más modell:** korotációs gerendaelemekkel ugyanilyen kis nyúlás/nagy elfordulás modell építhető. A jelen példa egy 2D kontinuum útja; 3D lengéshez más háló és megfelelő 3D operátor kell.

A végső mérnöki modellhez geometriai, anyagi, időbeli és hálóbeli ellenőrzés is szükséges. Ez a notebook a két HHT-callback és egy objektív, konzisztens 2D geometriai nemlinearitás működő mintája.